# Experiment No. 9: Text Classification and Text Generation Using Transformers

**Name:** Pradeep Manikandan D  
**Roll Number:** 24BAD088

## 1. Install and import libraries

In [1]:
%pip install -q transformers datasets accelerate evaluate scikit-learn sentencepiece

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    AutoModelForSeq2SeqLM,
    set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

c:\Users\PRADEEP MANIKANDAN D\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 2. Load and explore the IMDb review dataset

In [3]:
# A smaller subset keeps the demonstration manageable. Increase these values if resources allow.
MAX_TRAIN_SAMPLES = 1000
MAX_EVAL_SAMPLES = 300
MAX_LENGTH = 128

dataset = load_dataset("stanfordnlp/imdb")

train_data = dataset["train"].shuffle(seed=42).select(
    range(min(MAX_TRAIN_SAMPLES, len(dataset["train"])))
)
eval_data = dataset["test"].shuffle(seed=42).select(
    range(min(MAX_EVAL_SAMPLES, len(dataset["test"])))
)

print("Training samples:", len(train_data))
print("Evaluation samples:", len(eval_data))
print("Label names: 0 = Negative, 1 = Positive")
display(pd.DataFrame(train_data.select(range(min(5, len(train_data))))))

c:\Users\PRADEEP MANIKANDAN D\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PRADEEP MANIKANDAN D\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\PRADEEP MANIKANDAN D\AppData\Local\Programs\Python\Pytho

Training samples: 1000
Evaluation samples: 300
Label names: 0 = Negative, 1 = Positive


,text,label
0,There is no relation at all between Fortier an...,1
1,This movie is a great. The plot is very true t...,1
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0
3,In the process of trying to establish the audi...,1
4,"Yeh, I know -- you're quivering with excitemen...",0


## 3. Tokenize the text

In [4]:
bert_checkpoint = "google-bert/bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_checkpoint)

def tokenize_reviews(batch):
    return bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_data.map(tokenize_reviews, batched=True)
tokenized_eval = eval_data.map(tokenize_reviews, batched=True)

# Keep only the model inputs and labels.
columns_to_remove_train = [
    col for col in tokenized_train.column_names
    if col not in ["input_ids", "token_type_ids", "attention_mask", "label"]
]
columns_to_remove_eval = [
    col for col in tokenized_eval.column_names
    if col not in ["input_ids", "token_type_ids", "attention_mask", "label"]
]
tokenized_train = tokenized_train.remove_columns(columns_to_remove_train)
tokenized_eval = tokenized_eval.remove_columns(columns_to_remove_eval)

# Transformers sequence-classification models expect the target field to be named "labels".
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval = tokenized_eval.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)
print(tokenized_train[0].keys())

c:\Users\PRADEEP MANIKANDAN D\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PRADEEP MANIKANDAN D\.cache\huggingface\hub\models--google-bert--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 300/300 [00:00<00:00, 1180.32 ex

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


## 4. Fine-tune BERT for sentiment classification

In [5]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_checkpoint,
    num_labels=2,
    id2label={0: "Negative", 1: "Positive"},
    label2id={"Negative": 0, "Positive": 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="./bert-sentiment-results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_result = trainer.train()
print("Training complete.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 288.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## 5. Evaluate the classification model

In [ ]:
evaluation_results = trainer.evaluate()
print("Evaluation metrics:")
for metric, value in evaluation_results.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")

In [ ]:
prediction_output = trainer.predict(tokenized_eval)
predicted_labels = np.argmax(prediction_output.predictions, axis=-1)
true_labels = prediction_output.label_ids

print(classification_report(
    true_labels,
    predicted_labels,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

## 6. Test BERT with new text inputs

In [ ]:
def predict_sentiment(text):
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(bert_model.device)

    bert_model.eval()
    with torch.no_grad():
        logits = bert_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(torch.argmax(probabilities).item())
    label = "Positive" if predicted_id == 1 else "Negative"
    return {
        "text": text,
        "predicted_sentiment": label,
        "confidence": round(float(probabilities[predicted_id].item()), 4)
    }

test_reviews = [
    "The movie was wonderful, with excellent acting and a touching story.",
    "The plot was boring and the performances were disappointing.",
    "I enjoyed the film and would happily watch it again."
]

display(pd.DataFrame([predict_sentiment(text) for text in test_reviews]))

## 7. Save the fine-tuned BERT model and tokenizer

In [ ]:
bert_save_path = "./bert-sentiment-model"
trainer.save_model(bert_save_path)
bert_tokenizer.save_pretrained(bert_save_path)
print("Saved BERT model and tokenizer to:", bert_save_path)

## 8. Load T5-small for sequence-to-sequence text generation

In [ ]:
t5_checkpoint = "google-t5/t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_checkpoint)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_checkpoint).to(device)
t5_model.eval()

print("T5-small loaded on:", device)

## 9. Generate text using T5-small

In [ ]:
def generate_text(text, max_new_tokens=80, num_beams=4):
    # T5 uses task prefixes; "summarize:" requests a short summary.
    prompt = "summarize: " + text
    inputs = t5_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = t5_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )

    return t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)

sample_texts = [
    "Transformers use self-attention to process relationships between tokens. "
    "They are widely used in natural language processing tasks such as translation, "
    "summarization, classification, and question answering.",
    "Renewable energy sources such as solar and wind power can reduce dependence on "
    "fossil fuels. Energy storage and reliable electricity grids help integrate these "
    "sources into modern power systems."
]

generation_results = []
for text in sample_texts:
    generation_results.append({
        "input_text": text,
        "generated_summary": generate_text(text)
    })

display(pd.DataFrame(generation_results))

## 10. Experiment with generation settings

In [ ]:
example_text = (
    "A healthy ecosystem supports many kinds of plants and animals. "
    "Protecting forests, rivers, and wetlands helps maintain biodiversity and "
    "provides clean water and air for communities."
)

settings_results = []
for beams in [1, 4]:
    summary = generate_text(example_text, max_new_tokens=40, num_beams=beams)
    settings_results.append({
        "num_beams": beams,
        "max_new_tokens": 40,
        "generated_summary": summary
    })

display(pd.DataFrame(settings_results))

## 11. Save generated outputs

In [ ]:
output_df = pd.DataFrame(generation_results)
output_df.to_csv("t5_generated_summaries.csv", index=False)
print("Generated summaries saved to t5_generated_summaries.csv")
display(output_df)

## Notes
- Run the notebook in Google Colab with a GPU runtime for faster BERT fine-tuning.
- The training subset is intentionally small for a lab demonstration; results may vary with dataset sampling, runtime, and model versions.
- The first run downloads datasets and pretrained model weights, so an internet connection is required.
- Review generated summaries for relevance and factual consistency before using them.